# PS-S6E6 RAPIDS XGBoost GPU Model v5

This notebook trains a single 5-fold XGBoost model for the Playground Series S6E6 stellar classification competition. It reproduces local experiment `EXP3-110`, the strongest XGBoost experiment from my `exp-xgb3` search.

The model is GPU-first: feature engineering uses cuDF/CuPy, target encoding uses cuML, and XGBoost trains with CUDA histogram trees. The feature set is intentionally capped at 370 selected columns so it is practical on Kaggle's T4 GPU with 16GB VRAM. The original SDSS dataset is used only to create unsupervised/original-label prior features; original rows are not appended to the fold training data.

Validation is leak-free: the outer split is a deterministic stratified 5-fold split with seed 42, and target encoding is fit inside each training fold with inner out-of-fold encoding. No stacking, blending, or post-processing is used.


In [1]:
import gc, os
import glob
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import cupy as cp
import cudf
from cuml.preprocessing import TargetEncoder
import xgboost as xgb

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)


In [2]:
SEED = 42
N_SPLITS = 5
TARGET = 'class'
ID_COL = 'id'
MODEL_ID = 'xgb-5'

CLASSES = ['GALAXY', 'QSO', 'STAR']
CLASS_TO_INT = {c: i for i, c in enumerate(CLASSES)}
INT_TO_CLASS = {i: c for c, i in CLASS_TO_INT.items()}

EPS = 1e-6
RAW_NUM_COLS = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
BANDS = ['u', 'g', 'r', 'i', 'z']

# EXP3-110 configuration. Original labels are used for prior features only;
# original rows are not appended to fold training data.
USE_ORIGINAL_ROWS = False
USE_CLASS_WEIGHTS = True
CLASS_WEIGHT_POWER = 1.0

TE_SOURCE = 'all'
TE_SMOOTH = 16.0
TE_INNER_SPLITS = 7
TE_MAX_CARDINALITY = 5000
TOP_N_FEATURES = 370

ART_LOWFREQ_COLS = [f'art_{c}_floor' for c in RAW_NUM_COLS]
ART_LOWFREQ_PAIR_COLS = ['art_alpha_floor_x_delta_floor', 'art_u_floor_x_z_floor']
ART_COLOR_BIN_SPECS = [
    ('u_g', 2.0, 'half'),
    ('g_r', 2.0, 'half'),
    ('r_i', 2.0, 'half'),
    ('i_z', 2.0, 'half'),
    ('u_r', 1.0, 'one'),
    ('redshift', 10.0, 'tenth'),
    ('alpha', 0.2, 'deg5'),
    ('delta', 0.2, 'deg5'),
]
ART_COLOR_COLS = [f'art_{c}_{tag}' for c, _, tag in ART_COLOR_BIN_SPECS]
ART_COLOR_PAIR_COLS = [
    'art_u_g_half_x_redshift_tenth',
    'art_g_r_half_x_redshift_tenth',
    'art_alpha_deg5_x_delta_deg5',
]

OOF_PATH = Path('train_oof') / f'{MODEL_ID}_oof.npy'
PRED_PATH = Path('test_preds') / f'{MODEL_ID}_test_preds.npy'
SUB_PATH = Path('.') / f'{MODEL_ID}_submission.csv'
for path in [OOF_PATH.parent, PRED_PATH.parent, SUB_PATH.parent]:
    path.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
cp.random.seed(SEED)

## Load Data

In [3]:
def find_competition_root():
    candidates = [
        Path('/kaggle/input/competitions/playground-series-s6e6'),
    ]
    candidates += [Path(p).parent for p in glob.glob('/kaggle/input/*/train.csv')]
    for root in candidates:
        if (root / 'train.csv').exists() and (root / 'test.csv').exists():
            return root
    raise FileNotFoundError('Could not find train.csv and test.csv. Add the competition data to the notebook inputs.')


def find_original_path():
    candidates = [
        Path('/kaggle/input/datasets/fedesoriano/stellar-classification-dataset-sdss17/star_classification.csv'),
    ]
    candidates += [Path(p) for p in glob.glob('/kaggle/input/*/star_classification.csv')]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        'Could not find star_classification.csv. Add the original stellar classification dataset '
        'to the Kaggle notebook inputs for this model.'
    )


def clean_num(s):
    return cudf.to_numeric(s, errors='coerce').astype('float32')


def cat_key(s):
    return s.astype('str').fillna('__NA__')


def spectral_type(g, r):
    return cudf.cut(
        r - g,
        [-np.inf, -1, -0.5, 0, np.inf],
        labels=['M', 'G/K', 'A/F', 'O/B'],
    ).astype('str')


def galaxy_population(u, r):
    return cudf.cut(
        u - r,
        [-np.inf, 2.2, np.inf],
        labels=['Blue_Cloud', 'Red_Sequence'],
    ).astype('str')


def read_competition_csv(path):
    df = cudf.read_csv(str(path))
    for c in RAW_NUM_COLS:
        df[c] = clean_num(df[c])
    if ID_COL in df.columns:
        df[ID_COL] = df[ID_COL].astype('int32')
    df['spectral_type'] = cat_key(df['spectral_type'])
    df['galaxy_population'] = cat_key(df['galaxy_population'])
    return df


def read_original_csv(path):
    orig = cudf.read_csv(str(path))
    for c in RAW_NUM_COLS:
        orig[c] = clean_num(orig[c])
    if 'spectral_type' not in orig.columns:
        orig['spectral_type'] = spectral_type(orig['g'], orig['r'])
    if 'galaxy_population' not in orig.columns:
        orig['galaxy_population'] = galaxy_population(orig['u'], orig['r'])
    orig['spectral_type'] = cat_key(orig['spectral_type'])
    orig['galaxy_population'] = cat_key(orig['galaxy_population'])
    keep = RAW_NUM_COLS + ['spectral_type', 'galaxy_population', TARGET]
    return orig[[c for c in keep if c in orig.columns]].copy()


DATA_ROOT = find_competition_root()
ORIG_PATH = find_original_path()

train = read_competition_csv(DATA_ROOT / 'train.csv')
test = read_competition_csv(DATA_ROOT / 'test.csv')
orig = read_original_csv(ORIG_PATH)
sample_path = DATA_ROOT / 'sample_submission.csv'
sample = pd.read_csv(sample_path) if sample_path.exists() else None

y = train[TARGET].map(CLASS_TO_INT).astype('int8').reset_index(drop=True)
y_orig = orig[TARGET].map(CLASS_TO_INT).astype('int8').reset_index(drop=True)
test_ids = test[ID_COL].copy()

print('competition root:', DATA_ROOT)
print('original dataset:', ORIG_PATH)
print('train/test/original:', train.shape, test.shape, orig.shape)
print(train[TARGET].value_counts(normalize=True).sort_index().to_pandas())

competition root: /kaggle/input/competitions/playground-series-s6e6
original dataset: /kaggle/input/datasets/fedesoriano/stellar-classification-dataset-sdss17/star_classification.csv
train/test/original: (577347, 12) (247435, 11) (100000, 11)
class
GALAXY    0.653818
QSO       0.202899
STAR      0.143283
Name: proportion, dtype: float64


## Feature Engineering

In [4]:
def assign_cupy(df, name, values, dtype="float32"):
    df[name] = cudf.Series(values.astype(dtype), index=df.index)


def add_public_features(df, style="full"):
    out = df.copy()
    for c in RAW_NUM_COLS:
        out[c] = clean_num(out[c])

    color_pairs = [
        ("u", "g"), ("g", "r"), ("r", "i"), ("i", "z"),
        ("u", "r"), ("u", "i"), ("u", "z"), ("g", "i"),
        ("g", "z"), ("r", "z"),
    ]
    for a, b in color_pairs:
        out[f"{a}_{b}"] = (out[a] - out[b]).astype("float32")

    band_values = out[BANDS].to_cupy().astype(cp.float32)
    assign_cupy(out, "mag_mean", cp.mean(band_values, axis=1))
    assign_cupy(out, "mag_std", cp.std(band_values, axis=1, ddof=1))
    assign_cupy(out, "mag_min", cp.min(band_values, axis=1))
    assign_cupy(out, "mag_max", cp.max(band_values, axis=1))
    out["mag_range"] = (out["mag_max"] - out["mag_min"]).astype("float32")

    for b in BANDS:
        out[f"redshift_{b}"] = (out["redshift"] * out[b]).astype("float32")

    alpha_rad = out["alpha"].to_cupy().astype(cp.float32) * np.float32(np.pi / 180.0)
    delta_rad = out["delta"].to_cupy().astype(cp.float32) * np.float32(np.pi / 180.0)
    assign_cupy(out, "alpha_sin", cp.sin(alpha_rad))
    assign_cupy(out, "alpha_cos", cp.cos(alpha_rad))
    assign_cupy(out, "delta_sin", cp.sin(delta_rad))
    assign_cupy(out, "delta_cos", cp.cos(delta_rad))

    spectral_map = {"O/B": 0, "A": 1, "F": 2, "G": 3, "K": 4, "M": 5}
    out["spectral_ord"] = out["spectral_type"].map(spectral_map).fillna(-1).astype("float32")

    flux_arrays = []
    for b in BANDS:
        clipped = cp.clip(out[b].to_cupy().astype(cp.float32), -30, 30)
        flux = cp.power(cp.float32(10.0), cp.float32(-0.4) * clipped).astype(cp.float32)
        assign_cupy(out, f"flux_{b}", flux)
        flux_arrays.append(flux)
    flux_values = cp.vstack(flux_arrays).T
    assign_cupy(out, "flux_mean", cp.mean(flux_values, axis=1))
    assign_cupy(out, "flux_std", cp.std(flux_values, axis=1, ddof=1))
    assign_cupy(out, "flux_min", cp.min(flux_values, axis=1))
    assign_cupy(out, "flux_max", cp.max(flux_values, axis=1))
    out["flux_range"] = (out["flux_max"] - out["flux_min"]).astype("float32")

    x = cp.arange(len(BANDS), dtype=cp.float32)
    x_centered = x - x.mean()
    assign_cupy(out, "mag_slope", (band_values - cp.mean(band_values, axis=1, keepdims=True)).dot(x_centered) / cp.sum(x_centered ** 2))
    out["mag_curvature"] = (out["u"] - 2 * out["r"] + out["z"]).astype("float32")
    out["blue_curvature"] = (out["u"] - 2 * out["g"] + out["r"]).astype("float32")
    out["red_curvature"] = (out["r"] - 2 * out["i"] + out["z"]).astype("float32")

    for c in ["u_g", "g_r", "r_i", "i_z"]:
        out[f"{c}_per_redshift"] = (out[c] / (out["redshift"].abs() + EPS)).astype("float32")

    cat_cols = ["spectral_type", "galaxy_population"]

    if style == "full":
        out["mag_argmin"] = cudf.Series(cp.argmin(band_values, axis=1).astype(cp.int16), index=out.index)
        out["mag_argmax"] = cudf.Series(cp.argmax(band_values, axis=1).astype(cp.int16), index=out.index)
        redshift_cp = out["redshift"].to_cupy().astype(cp.float32)
        signed_redshift_denom = cp.where(
            cp.abs(redshift_cp) < EPS,
            cp.where(redshift_cp < 0, cp.float32(-EPS), cp.float32(EPS)),
            redshift_cp,
        )
        for b in BANDS:
            out[f"{b}_over_redshift"] = (out[b] / (out["redshift"].abs() + EPS)).astype("float32")
            assign_cupy(out, f"{b}_over_redshift_signed", out[b].to_cupy().astype(cp.float32) / signed_redshift_denom)
        assign_cupy(out, "sky_x", cp.cos(delta_rad) * cp.cos(alpha_rad))
        assign_cupy(out, "sky_y", cp.cos(delta_rad) * cp.sin(alpha_rad))
        assign_cupy(out, "sky_z", cp.sin(delta_rad))
        out["redshift_abs"] = out["redshift"].abs().astype("float32")
        assign_cupy(out, "redshift_sky_x", redshift_cp * out["sky_x"].to_cupy().astype(cp.float32))
        assign_cupy(out, "redshift_sky_y", redshift_cp * out["sky_y"].to_cupy().astype(cp.float32))
        assign_cupy(out, "redshift_sky_z", redshift_cp * out["sky_z"].to_cupy().astype(cp.float32))
        redshift_abs_cp = cp.abs(redshift_cp)
        assign_cupy(out, "redshift_abs_sky_x", redshift_abs_cp * out["sky_x"].to_cupy().astype(cp.float32))
        assign_cupy(out, "redshift_abs_sky_y", redshift_abs_cp * out["sky_y"].to_cupy().astype(cp.float32))
        assign_cupy(out, "redshift_abs_sky_z", redshift_abs_cp * out["sky_z"].to_cupy().astype(cp.float32))
        distmod_proxy = cp.float32(5.0) * cp.log10(redshift_abs_cp + EPS)
        assign_cupy(out, "redshift_distmod_proxy", distmod_proxy)
        for b in BANDS:
            assign_cupy(out, f"{b}_absmag_proxy", out[b].to_cupy().astype(cp.float32) - distmod_proxy)
        assign_cupy(out, "mag_mean_absmag_proxy", out["mag_mean"].to_cupy().astype(cp.float32) - distmod_proxy)
        out["redshift_log1p_abs"] = cudf.Series(cp.log1p(out["redshift_abs"].to_cupy()).astype(cp.float32), index=out.index)
        out["redshift_is_neg"] = (out["redshift"] < 0).astype("int8")
        phys_bins = cp.asarray([-np.inf, 0.05, 0.10, 0.30, 0.60, np.inf], dtype=cp.float32)
        phys_codes = cp.searchsorted(phys_bins, redshift_cp, side="right") - 1
        phys_codes = cp.where(cp.isnan(redshift_cp), -1, phys_codes)
        out["redshift_phys_bin"] = cudf.Series(phys_codes.astype(cp.int8), index=out.index).astype("str")
        gr_cp = out["g_r"].to_cupy().astype(cp.float32)
        gr_bins = cp.asarray([-np.inf, 0.0, 0.4, 0.8, 1.2, np.inf], dtype=cp.float32)
        gr_codes = cp.searchsorted(gr_bins, gr_cp, side="right") - 1
        gr_codes = cp.where(cp.isnan(gr_cp), -1, gr_codes)
        out["g_r_color_bin"] = cudf.Series(gr_codes.astype(cp.int8), index=out.index).astype("str")
        out["redshift_phys_x_g_r_color"] = cat_key(out["redshift_phys_bin"]) + "__" + cat_key(out["g_r_color_bin"])
        out["spectral_type_calc"] = spectral_type(out["g"], out["r"])
        out["galaxy_population_calc"] = galaxy_population(out["u"], out["r"])
        out["spectral_x_pop"] = cat_key(out["spectral_type"]) + "__" + cat_key(out["galaxy_population"])
        out["spectral_calc_x_pop_calc"] = cat_key(out["spectral_type_calc"]) + "__" + cat_key(out["galaxy_population_calc"])
        out["redshift_phys_x_spectral"] = cat_key(out["redshift_phys_bin"]) + "__" + cat_key(out["spectral_type"])
        out["redshift_phys_x_pop"] = cat_key(out["redshift_phys_bin"]) + "__" + cat_key(out["galaxy_population"])
        out["redshift_phys_x_spectral_pop"] = cat_key(out["redshift_phys_bin"]) + "__" + cat_key(out["spectral_x_pop"])
        for c in ["u_g", "g_r", "r_i", "i_z", "u_r", "g_i", "r_z"]:
            out[f"{c}_x_redshift"] = (out[c] * out["redshift"]).astype("float32")
            out[f"{c}_abs"] = out[c].abs().astype("float32")
            assign_cupy(out, f"{c}_over_redshift_signed", out[c].to_cupy().astype(cp.float32) / signed_redshift_denom)
        ug = out["u_g"].to_cupy().astype(cp.float32)
        gr = out["g_r"].to_cupy().astype(cp.float32)
        ri = out["r_i"].to_cupy().astype(cp.float32)
        iz = out["i_z"].to_cupy().astype(cp.float32)
        assign_cupy(out, "color_plane_radius_ug_gr", cp.sqrt(ug ** 2 + gr ** 2))
        assign_cupy(out, "color_plane_angle_ug_gr", cp.arctan2(ug, gr + EPS))
        assign_cupy(out, "color_plane_radius_ri_iz", cp.sqrt(ri ** 2 + iz ** 2))
        assign_cupy(out, "color_plane_angle_ri_iz", cp.arctan2(ri, iz + EPS))
        cat_cols = [
            "spectral_type", "galaxy_population", "spectral_type_calc", "galaxy_population_calc",
            "spectral_x_pop", "spectral_calc_x_pop_calc", "redshift_phys_bin", "g_r_color_bin",
            "redshift_phys_x_g_r_color", "redshift_phys_x_spectral", "redshift_phys_x_pop",
            "redshift_phys_x_spectral_pop",
        ]

    return out.replace([np.inf, -np.inf], np.nan), cat_cols


def qcut_codes_gpu(values, ref_values, q):
    ref = ref_values.dropna()
    if len(ref) < 2:
        return cp.full(len(values), -1, dtype=cp.int16)
    probs = cp.linspace(0, 1, q + 1, dtype=cp.float32)
    bins = cp.asarray(ref.quantile(probs).values, dtype=cp.float32)
    bins = cp.unique(bins)
    vals = values.to_cupy().astype(cp.float32)
    if len(bins) <= 1:
        return cp.full(len(vals), -1, dtype=cp.int16)
    codes = cp.searchsorted(bins, vals, side="left") - 1
    codes = cp.where(vals == bins[0], 0, codes)
    codes = cp.where((vals < bins[0]) | (vals > bins[-1]) | cp.isnan(vals), -1, codes)
    return cp.clip(codes, -1, len(bins) - 2).astype(cp.int16)


def add_lowfreq_artifact_features(df):
    out = df.copy()
    cat_cols = []
    for c in RAW_NUM_COLS:
        vals = clean_num(out[c]).to_cupy().astype(cp.float32)
        finite = ~cp.isnan(vals)
        floored = cp.floor(cp.where(finite, vals, cp.float32(0.0))).astype(cp.int32)
        floored = cp.where(finite, floored, cp.int32(-2147483648))
        name = f"art_{c}_floor"
        out[name] = cudf.Series(floored, index=out.index).astype("str")
        cat_cols.append(name)
    out["art_alpha_floor_x_delta_floor"] = cat_key(out["art_alpha_floor"]) + "__" + cat_key(out["art_delta_floor"])
    out["art_u_floor_x_z_floor"] = cat_key(out["art_u_floor"]) + "__" + cat_key(out["art_z_floor"])
    cat_cols.extend(ART_LOWFREQ_PAIR_COLS)
    return out, cat_cols


def add_color_artifact_features(df):
    out = df.copy()
    cat_cols = []
    for c, scale, tag in ART_COLOR_BIN_SPECS:
        if c not in out.columns:
            continue
        vals = clean_num(out[c]).to_cupy().astype(cp.float32)
        finite = ~cp.isnan(vals)
        binned = cp.floor(cp.where(finite, vals * np.float32(scale), cp.float32(0.0))).astype(cp.int32)
        binned = cp.where(finite, binned, cp.int32(-2147483648))
        name = f"art_{c}_{tag}"
        out[name] = cudf.Series(binned, index=out.index).astype("str")
        cat_cols.append(name)
    if "art_u_g_half" in out.columns and "art_redshift_tenth" in out.columns:
        out["art_u_g_half_x_redshift_tenth"] = cat_key(out["art_u_g_half"]) + "__" + cat_key(out["art_redshift_tenth"])
        cat_cols.append("art_u_g_half_x_redshift_tenth")
    if "art_g_r_half" in out.columns and "art_redshift_tenth" in out.columns:
        out["art_g_r_half_x_redshift_tenth"] = cat_key(out["art_g_r_half"]) + "__" + cat_key(out["art_redshift_tenth"])
        cat_cols.append("art_g_r_half_x_redshift_tenth")
    if "art_alpha_deg5" in out.columns and "art_delta_deg5" in out.columns:
        out["art_alpha_deg5_x_delta_deg5"] = cat_key(out["art_alpha_deg5"]) + "__" + cat_key(out["art_delta_deg5"])
        cat_cols.append("art_alpha_deg5_x_delta_deg5")
    return out, cat_cols


def add_quantile_bin_features(df, train_test_mask, extra_qbins=None):
    out = df.copy()
    qbin_cols = []
    cols = RAW_NUM_COLS + [c for c in ["u_g", "g_r", "r_i", "i_z", "u_r", "mag_mean", "mag_range"] if c in out.columns]
    qbins = sorted(set([16, 64, 256] + list(extra_qbins or [])))
    for c in cols:
        s = clean_num(out[c])
        ref = s[train_test_mask]
        for q in qbins:
            name = f"{c}_qbin{q}"
            out[name] = cudf.Series(qcut_codes_gpu(s, ref, q), index=out.index).astype("int16").astype("str")
            qbin_cols.append(name)
    for a, b in [("alpha_qbin64", "delta_qbin64"), ("u_g_qbin64", "g_r_qbin64"), ("redshift_qbin64", "mag_mean_qbin64")]:
        if a in out.columns and b in out.columns:
            name = f"{a}__x__{b}"
            out[name] = cat_key(out[a]) + "__" + cat_key(out[b])
            qbin_cols.append(name)
    return out, qbin_cols


def select_te_cols(df, cat_cols, source, max_card):
    cols = []
    for c in cat_cols:
        if c not in df.columns:
            continue
        card = int(cat_key(df[c]).nunique(dropna=False))
        if card > max_card:
            continue
        if source == "core":
            keep = (
                c in ["spectral_type", "galaxy_population", "spectral_type_calc", "galaxy_population_calc", "spectral_x_pop", "spectral_calc_x_pop_calc"]
                or c.endswith("_qbin16")
                or c.endswith("_qbin64")
                or "_rkey" in c
                or "__x__" in c
            )
        elif source == "qbin16":
            keep = c.endswith("_qbin16") or c in ["spectral_type", "galaxy_population", "spectral_type_calc", "galaxy_population_calc"]
        else:
            keep = True
        if keep:
            cols.append(c)
    return cols


def add_frequency_features(df, cols, fit_mask):
    out = df.copy()
    for c in cols:
        s = cat_key(out[c])
        vc = s[fit_mask].value_counts(dropna=False)
        out[f"{c}_freq"] = s.map(vc).fillna(0).astype("float32")
        out[f"{c}_freq_log1p"] = cudf.Series(cp.log1p(out[f"{c}_freq"].to_cupy()).astype(cp.float32), index=out.index)
    return out


def add_original_prior_features(df, cols, orig_mask, orig_y, smooth=0.0):
    out = df.copy()
    prior_counts = cp.bincount(orig_y.to_cupy().astype(cp.int32), minlength=len(CLASSES)).astype(cp.float32)
    prior = prior_counts / cp.maximum(prior_counts.sum(), cp.float32(1.0))
    smooth = float(smooth or 0.0)
    smooth_tag = int(round(smooth)) if smooth else 0
    for c in cols:
        key = cat_key(out[c])
        tmp = cudf.DataFrame({"key": key[orig_mask].reset_index(drop=True), "y": orig_y.reset_index(drop=True)})
        counts = tmp.groupby("key").size()
        out[f"orig_{c}_count"] = key.map(counts).fillna(0).astype("float32")
        for cls_idx, cls_name in INT_TO_CLASS.items():
            hit = (tmp["y"] == cls_idx).astype("float32")
            rates = tmp.assign(hit=hit).groupby("key")["hit"].mean()
            out[f"orig_{c}_prior_{cls_name}"] = key.map(rates).fillna(float(prior[cls_idx].get())).astype("float32")
            if smooth > 0:
                smooth_rates = ((rates * counts.astype("float32")) + np.float32(smooth) * np.float32(float(prior[cls_idx].get()))) / (counts.astype("float32") + np.float32(smooth))
                out[f"orig_{c}_smooth{smooth_tag}_prior_{cls_name}"] = key.map(smooth_rates).fillna(float(prior[cls_idx].get())).astype("float32")
    return out

In [5]:
def build_feature_matrix(train, test, orig):
    train_base = train.drop(columns=[TARGET]).copy()
    test_base = test.copy()
    orig_base = orig.drop(columns=[TARGET]).copy()
    train_base['_source'] = 'train'
    test_base['_source'] = 'test'
    orig_base['_source'] = 'orig'

    all_df = cudf.concat([train_base, test_base, orig_base], axis=0, ignore_index=True)
    all_df, cat_cols = add_public_features(all_df, 'full')
    train_test_mask = all_df['_source'].isin(['train', 'test'])

    all_df, artifact_cols = add_lowfreq_artifact_features(all_df)
    cat_cols += artifact_cols
    all_df, artifact_cols = add_color_artifact_features(all_df)
    cat_cols += artifact_cols
    all_df, qbin_cols = add_quantile_bin_features(all_df, train_test_mask, extra_qbins=[])
    cat_cols += qbin_cols

    cat_cols = [c for c in dict.fromkeys(cat_cols) if c in all_df.columns]

    freq_cols = select_te_cols(all_df, cat_cols, TE_SOURCE, max_card=TE_MAX_CARDINALITY * 4)
    all_df = add_frequency_features(all_df, freq_cols, all_df['_source'].isin(['train', 'test', 'orig']))

    orig_mask = all_df['_source'].eq('orig')
    prior_cols = select_te_cols(all_df, cat_cols, TE_SOURCE, max_card=TE_MAX_CARDINALITY * 2)
    all_df = add_original_prior_features(all_df, prior_cols, orig_mask, y_orig, smooth=0.0)

    all_df['is_orig'] = all_df['_source'].eq('orig').astype('int8')
    all_df['is_test'] = all_df['_source'].eq('test').astype('int8')
    all_df = all_df.drop(columns=[c for c in [ID_COL, '_source'] if c in all_df.columns]).replace([np.inf, -np.inf], np.nan)

    n_train = len(train_base)
    n_test = len(test_base)
    X = all_df.iloc[:n_train].reset_index(drop=True)
    X_test = all_df.iloc[n_train:n_train + n_test].reset_index(drop=True)
    X_orig = all_df.iloc[n_train + n_test:].reset_index(drop=True)
    cat_cols = [c for c in cat_cols if c in X.columns]

    del all_df, train_base, test_base, orig_base
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    return X, X_test, X_orig, cat_cols


X, X_test, X_orig, cat_cols = build_feature_matrix(train, test, orig)
print('base matrices:', X.shape, X_test.shape, X_orig.shape, 'cat_cols:', len(cat_cols))

del train, test, orig
gc.collect()
cp.get_default_memory_pool().free_all_blocks()

base matrices: (577347, 672) (247435, 672) (100000, 672) cat_cols: 81


## Selected Features

In [6]:
TOP_FEATURES = ['TE_art_g_r_half_x_redshift_tenth_STAR',
 'TE_art_g_r_half_x_redshift_tenth_QSO',
 'orig_art_g_r_half_x_redshift_tenth_prior_QSO',
 'redshift_u',
 'z_over_redshift',
 'TE_art_u_g_half_x_redshift_tenth_QSO',
 'TE_art_g_r_half_x_redshift_tenth_GALAXY',
 'g_z',
 'g_i',
 'orig_g_qbin64_prior_QSO',
 'redshift_g',
 'orig_g_qbin16_prior_QSO',
 'i_over_redshift',
 'redshift_abs',
 'TE_redshift_qbin16_GALAXY',
 'redshift_log1p_abs',
 'u_over_redshift',
 'TE_redshift_qbin64__x__mag_mean_qbin64_GALAXY',
 'g_i_abs',
 'orig_art_g_floor_prior_QSO',
 'u_i',
 'TE_redshift_qbin64_GALAXY',
 'orig_art_u_floor_x_z_floor_prior_QSO',
 'TE_alpha_qbin64__x__delta_qbin64_STAR',
 'art_redshift_floor_freq',
 'g_over_redshift',
 'u_r_abs',
 'redshift_is_neg',
 'orig_g_qbin256_prior_QSO',
 'TE_art_redshift_tenth_GALAXY',
 'TE_alpha_qbin64__x__delta_qbin64_GALAXY',
 'TE_redshift_qbin64_QSO',
 'art_redshift_floor',
 'orig_art_u_g_half_x_redshift_tenth_prior_QSO',
 'u_r',
 'mag_slope',
 'art_g_r_half_x_redshift_tenth_freq_log1p',
 'r_over_redshift',
 'g_qbin16',
 'TE_art_u_floor_x_z_floor_QSO',
 'TE_u_g_qbin64__x__g_r_qbin64_STAR',
 'orig_redshift_qbin64_prior_STAR',
 'flux_std',
 'redshift',
 'TE_art_u_g_half_x_redshift_tenth_GALAXY',
 'orig_redshift_qbin64_prior_GALAXY',
 'orig_g_qbin16_prior_GALAXY',
 'flux_g',
 'orig_redshift_qbin64__x__mag_mean_qbin64_prior_GALAXY',
 'mag_std',
 'TE_art_u_floor_x_z_floor_GALAXY',
 'redshift_z',
 'orig_redshift_qbin64__x__mag_mean_qbin64_prior_QSO',
 'g',
 'orig_redshift_qbin256_prior_GALAXY',
 'orig_art_i_floor_prior_QSO',
 'flux_range',
 'orig_art_redshift_floor_prior_GALAXY',
 'orig_art_r_floor_prior_QSO',
 'orig_alpha_qbin64__x__delta_qbin64_prior_STAR',
 'art_r_floor',
 'TE_redshift_qbin64_STAR',
 'TE_g_qbin64_QSO',
 'art_g_r_half_x_redshift_tenth_freq',
 'orig_alpha_qbin64__x__delta_qbin64_prior_GALAXY',
 'orig_art_i_floor_count',
 'TE_u_r_qbin64_GALAXY',
 'u_g',
 'TE_g_qbin16_QSO',
 'g_r_x_redshift',
 'z',
 'redshift_r',
 'r_z',
 'art_g_r_half_x_redshift_tenth',
 'mag_range',
 'i',
 'TE_art_u_g_half_x_redshift_tenth_STAR',
 'art_g_floor',
 'flux_i',
 'TE_art_redshift_tenth_STAR',
 'orig_r_qbin16_prior_QSO',
 'redshift_i',
 'r_z_x_redshift',
 'flux_z',
 'g_i_x_redshift',
 'orig_art_u_floor_x_z_floor_prior_GALAXY',
 'r',
 'TE_i_qbin64_QSO',
 'r_z_abs',
 'orig_g_qbin64_prior_GALAXY',
 'TE_redshift_qbin64__x__mag_mean_qbin64_QSO',
 'art_redshift_floor_freq_log1p',
 'orig_i_qbin16_prior_QSO',
 'orig_art_u_r_one_prior_QSO',
 'orig_mag_range_qbin16_prior_STAR',
 'orig_art_alpha_deg5_x_delta_deg5_prior_GALAXY',
 'TE_redshift_qbin16_STAR',
 'orig_art_redshift_tenth_prior_GALAXY',
 'TE_art_alpha_deg5_x_delta_deg5_STAR',
 'flux_r',
 'orig_art_z_floor_count',
 'TE_u_r_qbin64_QSO',
 'orig_art_alpha_deg5_x_delta_deg5_prior_STAR',
 'orig_u_qbin16_prior_QSO',
 'r_i_x_redshift',
 'art_i_floor',
 'orig_mag_range_qbin64_prior_STAR',
 'orig_z_qbin16_prior_QSO',
 'orig_redshift_qbin16_prior_GALAXY',
 'art_u_g_half',
 'art_z_floor_freq',
 'TE_art_g_floor_QSO',
 'art_r_i_half_freq_log1p',
 'orig_mag_range_qbin16_prior_GALAXY',
 'mag_max',
 'flux_min',
 'TE_g_qbin64_GALAXY',
 'orig_art_u_g_half_x_redshift_tenth_prior_STAR',
 'redshift_qbin64__x__mag_mean_qbin64',
 'TE_alpha_qbin64__x__delta_qbin64_QSO',
 'orig_i_qbin256_prior_QSO',
 'art_u_g_half_x_redshift_tenth_freq_log1p',
 'orig_art_i_floor_prior_GALAXY',
 'TE_mag_range_qbin64_QSO',
 'orig_art_r_i_half_count',
 'art_z_floor_freq_log1p',
 'color_plane_radius_ug_gr',
 'art_g_r_half',
 'art_r_i_half_freq',
 'TE_g_qbin16_GALAXY',
 'TE_redshift_qbin64__x__mag_mean_qbin64_STAR',
 'TE_art_r_floor_QSO',
 'flux_u',
 'flux_max',
 'orig_i_qbin64_prior_QSO',
 'u',
 'TE_art_alpha_deg5_x_delta_deg5_GALAXY',
 'flux_mean',
 'orig_u_g_qbin16_prior_STAR',
 'u_g_abs',
 'orig_art_u_floor_prior_QSO',
 'mag_mean_qbin16',
 'TE_art_alpha_floor_STAR',
 'art_u_floor_x_z_floor',
 'u_z',
 'redshift_qbin64__x__mag_mean_qbin64_freq_log1p',
 'orig_art_alpha_deg5_x_delta_deg5_prior_QSO',
 'orig_art_g_r_half_x_redshift_tenth_prior_STAR',
 'orig_mag_range_qbin256_prior_QSO',
 'mag_min',
 'orig_u_r_qbin16_prior_QSO',
 'orig_art_redshift_tenth_count',
 'r_i',
 'redshift_qbin16',
 'TE_i_qbin16_QSO',
 'TE_u_g_qbin64_STAR',
 'art_u_g_half_x_redshift_tenth_freq',
 'TE_art_r_floor_GALAXY',
 'redshift_qbin64__x__mag_mean_qbin64_freq',
 'TE_mag_range_qbin64_GALAXY',
 'u_qbin16',
 'art_i_floor_freq',
 'mag_range_qbin256',
 'orig_mag_range_qbin256_prior_STAR',
 'art_i_floor_freq_log1p',
 'art_u_g_half_x_redshift_tenth',
 'orig_z_qbin64_prior_QSO',
 'art_r_floor_freq_log1p',
 'alpha_sin',
 'orig_art_r_floor_prior_GALAXY',
 'TE_r_i_qbin64_QSO',
 'orig_u_qbin16_prior_STAR',
 'color_plane_radius_ri_iz',
 'TE_art_alpha_deg5_x_delta_deg5_QSO',
 'delta_cos',
 'TE_r_i_qbin16_GALAXY',
 'art_g_floor_freq',
 'orig_art_u_g_half_x_redshift_tenth_count',
 'orig_delta_qbin256_prior_STAR',
 'TE_r_i_qbin16_QSO',
 'orig_art_u_floor_x_z_floor_prior_STAR',
 'mag_range_qbin16',
 'art_delta_deg5',
 'TE_u_g_qbin16_STAR',
 'u_r_x_redshift',
 'sky_y',
 'TE_r_i_qbin64_GALAXY',
 'art_g_floor_freq_log1p',
 'orig_art_alpha_floor_prior_STAR',
 'orig_delta_qbin256_prior_GALAXY',
 'u_g_x_redshift',
 'g_r',
 'r_qbin16',
 'TE_z_qbin64_QSO',
 'art_redshift_tenth_freq_log1p',
 'alpha_qbin256',
 'orig_i_qbin16_prior_GALAXY',
 'orig_art_alpha_floor_prior_GALAXY',
 'orig_art_r_i_half_prior_GALAXY',
 'orig_art_g_floor_prior_GALAXY',
 'orig_redshift_qbin256_count',
 'TE_g_qbin64_STAR',
 'orig_u_qbin16_count',
 'art_redshift_tenth_freq',
 'z_qbin16',
 'r_qbin64',
 'TE_u_qbin16_QSO',
 'art_u_r_one_freq_log1p',
 'orig_u_qbin64_prior_QSO',
 'orig_art_alpha_deg5_prior_STAR',
 'g_qbin64',
 'TE_r_qbin16_QSO',
 'i_qbin16',
 'orig_z_qbin256_prior_QSO',
 'orig_art_delta_deg5_prior_GALAXY',
 'orig_g_qbin256_prior_GALAXY',
 'blue_curvature',
 'TE_u_qbin64_QSO',
 'orig_art_u_g_half_x_redshift_tenth_prior_GALAXY',
 'TE_art_redshift_tenth_QSO',
 'alpha',
 'art_u_r_one_freq',
 'orig_mag_range_qbin256_prior_GALAXY',
 'delta_sin',
 'art_redshift_tenth',
 'TE_art_alpha_floor_GALAXY',
 'art_alpha_floor',
 'delta',
 'TE_art_u_r_one_GALAXY',
 'color_plane_angle_ug_gr',
 'orig_art_redshift_tenth_prior_QSO',
 'orig_art_r_i_half_prior_QSO',
 'orig_art_g_floor_prior_STAR',
 'mag_curvature',
 'TE_art_u_floor_x_z_floor_STAR',
 'art_delta_floor',
 'u_qbin16_freq_log1p',
 'TE_g_r_qbin64_GALAXY',
 'TE_r_qbin64_STAR',
 'sky_z',
 'art_u_floor_freq',
 'mag_mean',
 'g_r_abs',
 'TE_z_qbin16_QSO',
 'sky_x',
 'u_qbin16_freq',
 'TE_u_qbin64_STAR',
 'orig_art_u_floor_count',
 'art_alpha_floor_x_delta_floor',
 'art_u_floor_freq_log1p',
 'orig_z_qbin16_count',
 'art_u_r_one',
 'art_u_g_half_freq',
 'TE_redshift_qbin16_QSO',
 'orig_alpha_qbin64__x__delta_qbin64_prior_QSO',
 'z_qbin16_freq',
 'TE_u_g_qbin64_QSO',
 'TE_u_g_qbin64_GALAXY',
 'orig_g_qbin64_prior_STAR',
 'orig_art_g_r_half_prior_GALAXY',
 'TE_g_qbin16_STAR',
 'orig_art_g_r_half_x_redshift_tenth_count',
 'orig_art_delta_floor_prior_QSO',
 'orig_art_u_r_one_count',
 'orig_spectral_x_pop_prior_QSO',
 'redshift_qbin256',
 'orig_art_u_g_half_prior_STAR',
 'TE_g_r_qbin64_STAR',
 'orig_art_u_r_one_prior_STAR',
 'art_delta_floor_freq_log1p',
 'TE_art_alpha_floor_QSO',
 'art_delta_floor_freq',
 'art_delta_deg5_freq',
 'art_u_g_half_freq_log1p',
 'art_delta_deg5_freq_log1p',
 'z_qbin16_freq_log1p',
 'TE_u_r_qbin16_GALAXY',
 'TE_art_i_floor_GALAXY',
 'orig_art_alpha_floor_prior_QSO',
 'orig_art_alpha_deg5_prior_GALAXY',
 'orig_art_u_g_half_count',
 'orig_art_g_floor_count',
 'TE_art_i_floor_QSO',
 'art_alpha_deg5_freq_log1p',
 'art_r_floor_freq',
 'art_alpha_deg5_freq',
 'orig_art_alpha_deg5_count',
 'TE_art_g_floor_GALAXY',
 'orig_u_r_qbin64_prior_QSO',
 'TE_u_g_qbin64__x__g_r_qbin64_QSO',
 'orig_art_u_floor_prior_STAR',
 'art_alpha_deg5_x_delta_deg5',
 'orig_art_delta_floor_prior_GALAXY',
 'art_alpha_deg5',
 'orig_art_delta_deg5_count',
 'orig_u_r_qbin256_prior_QSO',
 'orig_art_alpha_deg5_prior_QSO',
 'orig_art_g_r_half_prior_QSO',
 'TE_art_alpha_deg5_STAR',
 'orig_art_delta_floor_count',
 'TE_g_r_qbin64_QSO',
 'art_alpha_deg5_x_delta_deg5_freq',
 'art_u_floor',
 'art_alpha_deg5_x_delta_deg5_freq_log1p',
 'orig_art_delta_deg5_prior_QSO',
 'art_alpha_floor_freq_log1p',
 'art_r_i_half',
 'orig_art_r_i_half_prior_STAR',
 'orig_u_g_qbin64_prior_STAR',
 'art_alpha_floor_freq',
 'orig_art_g_r_half_x_redshift_tenth_prior_GALAXY',
 'TE_art_alpha_deg5_GALAXY',
 'orig_art_u_g_half_prior_GALAXY',
 'orig_art_u_floor_x_z_floor_count',
 'orig_art_alpha_floor_count',
 'TE_art_delta_floor_QSO',
 'TE_art_u_floor_QSO',
 'TE_art_delta_deg5_GALAXY',
 'orig_art_alpha_deg5_x_delta_deg5_count',
 'orig_art_i_floor_prior_STAR',
 'orig_art_redshift_floor_prior_QSO',
 'TE_art_delta_floor_GALAXY',
 'TE_art_alpha_deg5_QSO',
 'orig_alpha_qbin256_prior_GALAXY',
 'TE_u_g_qbin64__x__g_r_qbin64_GALAXY',
 'orig_art_delta_deg5_prior_STAR',
 'art_u_floor_x_z_floor_freq_log1p',
 'TE_art_r_floor_STAR',
 'orig_art_z_floor_prior_GALAXY',
 'TE_art_u_g_half_GALAXY',
 'TE_art_g_floor_STAR',
 'TE_art_redshift_floor_STAR',
 'orig_art_r_floor_count',
 'TE_art_r_i_half_STAR',
 'art_u_floor_x_z_floor_freq',
 'TE_art_i_floor_STAR',
 'TE_art_delta_deg5_QSO',
 'TE_art_u_floor_STAR',
 'orig_art_delta_floor_prior_STAR',
 'TE_art_r_i_half_GALAXY',
 'TE_art_u_g_half_QSO',
 'orig_art_u_g_half_prior_QSO',
 'art_alpha_floor_x_delta_floor_freq',
 'orig_art_r_floor_prior_STAR',
 'orig_art_i_z_half_prior_QSO',
 'TE_art_delta_deg5_STAR',
 'orig_alpha_qbin256_prior_STAR',
 'art_alpha_floor_x_delta_floor_freq_log1p',
 'TE_art_z_floor_STAR',
 'orig_art_z_floor_prior_QSO',
 'orig_art_u_floor_prior_GALAXY',
 'orig_art_u_r_one_prior_GALAXY',
 'TE_art_g_r_half_QSO',
 'art_g_r_half_freq_log1p',
 'TE_art_u_r_one_QSO',
 'TE_art_u_g_half_STAR',
 'TE_art_r_i_half_QSO',
 'art_z_floor',
 'TE_art_z_floor_GALAXY',
 'TE_art_z_floor_QSO',
 'TE_art_u_floor_GALAXY',
 'TE_art_g_r_half_GALAXY',
 'TE_art_delta_floor_STAR',
 'orig_art_g_r_half_prior_STAR',
 'TE_art_g_r_half_STAR',
 'TE_art_u_r_one_STAR',
 'orig_art_z_floor_prior_STAR',
 'TE_art_redshift_floor_GALAXY',
 'TE_art_redshift_floor_QSO',
 'TE_art_i_z_half_QSO']

print(f'Selected {len(TOP_FEATURES)} top features from EXP3-030 gain rerank.')

Selected 370 top features from EXP3-030 gain rerank.


## Fold-Safe Target Encoding

In [7]:
def sorted_factorize_three(train_s, valid_s, test_s):
    vals = cudf.concat([cat_key(train_s), cat_key(valid_s), cat_key(test_s)], ignore_index=True)
    cats = vals.drop_duplicates().sort_values(ignore_index=True)
    mapper = cudf.Series(cp.arange(len(cats), dtype=cp.int32), index=cats)
    codes = vals.map(mapper).fillna(-1).astype('int32').reset_index(drop=True)
    n_tr, n_va = len(train_s), len(valid_s)
    return (
        codes.iloc[:n_tr].reset_index(drop=True),
        codes.iloc[n_tr:n_tr + n_va].reset_index(drop=True),
        codes.iloc[n_tr + n_va:].reset_index(drop=True),
    )


def make_inner_fold_ids(y_train):
    y_cpu = y_train.to_numpy()
    fold_ids = np.empty(len(y_cpu), dtype=np.int32)
    inner = StratifiedKFold(n_splits=TE_INNER_SPLITS, shuffle=True, random_state=SEED + 177)
    for fold_id, (_, va_idx) in enumerate(inner.split(np.zeros(len(y_cpu), dtype=np.int8), y_cpu)):
        fold_ids[va_idx] = fold_id
    return cp.asarray(fold_ids, dtype=cp.int32)


def add_fold_safe_te_gpu(X_train, y_train, X_valid, X_test_fold, te_cols):
    if not te_cols:
        return X_train, X_valid, X_test_fold, []
    X_train = X_train.copy()
    X_valid = X_valid.copy()
    X_test_fold = X_test_fold.copy()
    fold_ids = make_inner_fold_ids(y_train)
    y_cp = y_train.to_cupy().astype(cp.int32)
    added = []

    for c in te_cols:
        if c not in X_train.columns:
            continue
        tr_codes, va_codes, te_codes = sorted_factorize_three(X_train[c], X_valid[c], X_test_fold[c])
        tr_cp = tr_codes.to_cupy().astype(cp.int32)
        va_cp = va_codes.to_cupy().astype(cp.int32)
        te_cp = te_codes.to_cupy().astype(cp.int32)
        for cls_idx, cls_name in INT_TO_CLASS.items():
            y_bin = (y_cp == cls_idx).astype(cp.float32)
            encoder = TargetEncoder(
                n_folds=TE_INNER_SPLITS,
                smooth=TE_SMOOTH,
                seed=SEED + 177,
                split_method='customize',
                output_type='cupy',
            )
            tr_vals = cp.asarray(encoder.fit_transform(tr_cp, y_bin, fold_ids=fold_ids)).ravel().astype(cp.float32)
            va_vals = cp.asarray(encoder.transform(va_cp)).ravel().astype(cp.float32)
            te_vals = cp.asarray(encoder.transform(te_cp)).ravel().astype(cp.float32)
            name = f'TE_{c}_{cls_name}'
            X_train[name] = cudf.Series(tr_vals)
            X_valid[name] = cudf.Series(va_vals)
            X_test_fold[name] = cudf.Series(te_vals)
            added.append(name)
            del encoder, tr_vals, va_vals, te_vals, y_bin
        del tr_codes, va_codes, te_codes, tr_cp, va_cp, te_cp
    cp.get_default_memory_pool().free_all_blocks()
    return X_train, X_valid, X_test_fold, added


def encode_model_categories_gpu(X_train, X_valid, X_test_fold, model_cat_cols):
    X_train = X_train.copy()
    X_valid = X_valid.copy()
    X_test_fold = X_test_fold.copy()
    for c in model_cat_cols:
        if c not in X_train.columns:
            continue
        tr_codes, va_codes, te_codes = sorted_factorize_three(X_train[c], X_valid[c], X_test_fold[c])
        X_train[c] = tr_codes
        X_valid[c] = va_codes
        X_test_fold[c] = te_codes
    return X_train, X_valid, X_test_fold


def te_sources_needed_for_top_features(top_features, available_te_cols):
    needed = []
    for c in available_te_cols:
        prefix = f'TE_{c}_'
        if any(str(f).startswith(prefix) for f in top_features):
            needed.append(c)
    return needed


available_te_cols = select_te_cols(X, cat_cols, TE_SOURCE, TE_MAX_CARDINALITY)
TE_COLS = te_sources_needed_for_top_features(TOP_FEATURES, available_te_cols)
MODEL_CAT_COLS = [c for c in cat_cols if c in TOP_FEATURES]
print(f'target-encoding sources pruned: {len(available_te_cols)} -> {len(TE_COLS)}')
print(f'raw categorical features selected for model: {len(MODEL_CAT_COLS)}')
print(TE_COLS[:25])

target-encoding sources pruned: 80 -> 44
raw categorical features selected for model: 34
['art_alpha_floor', 'art_delta_floor', 'art_u_floor', 'art_g_floor', 'art_r_floor', 'art_i_floor', 'art_z_floor', 'art_redshift_floor', 'art_u_floor_x_z_floor', 'art_u_g_half', 'art_g_r_half', 'art_r_i_half', 'art_i_z_half', 'art_u_r_one', 'art_redshift_tenth', 'art_alpha_deg5', 'art_delta_deg5', 'art_u_g_half_x_redshift_tenth', 'art_g_r_half_x_redshift_tenth', 'art_alpha_deg5_x_delta_deg5', 'alpha_qbin64', 'u_qbin16', 'u_qbin64', 'g_qbin16', 'g_qbin64']


## Train XGBoost

In [8]:
def to_numpy(a):
    if isinstance(a, cp.ndarray):
        return cp.asnumpy(a)
    return np.asarray(a)


def balanced_error_metric(y_true, y_pred, sample_weight=None):
    y_true_np = to_numpy(y_true).astype(int)
    y_pred_np = to_numpy(y_pred)
    if y_pred_np.ndim == 1:
        y_pred_np = y_pred_np.reshape(-1, len(CLASSES))
    return 1.0 - balanced_accuracy_score(y_true_np, np.argmax(y_pred_np, axis=1))


def make_xgb_params(seed):
    return {
        'objective': 'multi:softprob',
        'num_class': len(CLASSES),
        'eval_metric': balanced_error_metric,
        'tree_method': 'hist',
        'device': 'cuda',
        'learning_rate': 0.012,
        'n_estimators': 7000,
        'early_stopping_rounds': 180,
        'max_depth': 0,
        'max_leaves': 72,
        'grow_policy': 'lossguide',
        'max_bin': 960,
        'min_child_weight': 10,
        'gamma': 0.2,
        'reg_alpha': 0.30,
        'reg_lambda': 4.0,
        'subsample': 0.82,
        'colsample_bytree': 0.74,
        'colsample_bylevel': 0.86,
        'random_state': seed,
        'n_jobs': 4,
    }


def class_weights_gpu(y_series):
    y_cp = y_series.to_cupy().astype(cp.int32)
    counts = cp.bincount(y_cp, minlength=len(CLASSES)).astype(cp.float32)
    weights_per_class = cp.float32(len(y_cp)) / (cp.float32(len(CLASSES)) * cp.maximum(counts, cp.float32(1.0)))
    weights = weights_per_class[y_cp]
    if CLASS_WEIGHT_POWER != 1.0:
        weights = cp.power(weights, CLASS_WEIGHT_POWER).astype(cp.float32)
    return weights.astype(cp.float32)


def prepare_selected_cupy(X_train, X_valid, X_test_fold):
    X_train, X_valid, X_test_fold = encode_model_categories_gpu(X_train, X_valid, X_test_fold, MODEL_CAT_COLS)
    missing = [c for c in TOP_FEATURES if c not in X_train.columns]
    if missing:
        print(f'Missing {len(missing)} top features; first missing:', missing[:10])
    features = [c for c in TOP_FEATURES if c in X_train.columns]
    X_train_cp = X_train[features].astype('float32').to_cupy().astype(cp.float32)
    X_valid_cp = X_valid[features].astype('float32').to_cupy().astype(cp.float32)
    X_test_cp = X_test_fold[features].astype('float32').to_cupy().astype(cp.float32)
    return X_train_cp, X_valid_cp, X_test_cp, features

In [9]:
y_cpu = y.to_numpy()
oof = np.zeros((len(X), len(CLASSES)), dtype='float32')
oof_fold = np.full(len(X), -1, dtype='int8')
test_pred_sum = np.zeros((len(X_test), len(CLASSES)), dtype='float32')
fold_rows = []
importance_rows = []

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_cpu), dtype=np.int8), y_cpu), start=1):
    fold_seed = SEED + fold * 100
    print(f'\n===== Fold {fold}/{N_SPLITS} | seed={fold_seed} =====')

    X_tr = X.iloc[tr_idx].reset_index(drop=True)
    y_tr = y.iloc[tr_idx].reset_index(drop=True)
    X_va = X.iloc[va_idx].reset_index(drop=True)
    y_va = y.iloc[va_idx].reset_index(drop=True)
    X_te = X_test.copy(deep=True)

    if USE_ORIGINAL_ROWS:
        raise ValueError('EXP3-110 should not append original rows. Keep USE_ORIGINAL_ROWS=False.')

    X_tr, X_va, X_te, added_te = add_fold_safe_te_gpu(X_tr, y_tr, X_va, X_te, TE_COLS)
    X_tr_cp, X_va_cp, X_te_cp, features = prepare_selected_cupy(X_tr, X_va, X_te)
    y_tr_cp = y_tr.to_cupy().astype(cp.int32)
    y_va_cp = y_va.to_cupy().astype(cp.int32)
    print('training shape:', X_tr_cp.shape, 'validation shape:', X_va_cp.shape, 'test shape:', X_te_cp.shape)
    print('TE features added before top selection:', len(added_te))

    sample_weight = class_weights_gpu(y_tr) if USE_CLASS_WEIGHTS else None
    valid_weight = class_weights_gpu(y_va) if USE_CLASS_WEIGHTS else None

    model = xgb.XGBClassifier(**make_xgb_params(fold_seed))
    model.fit(
        X_tr_cp,
        y_tr_cp,
        sample_weight=sample_weight,
        eval_set=[(X_va_cp, y_va_cp)],
        sample_weight_eval_set=[valid_weight] if valid_weight is not None else None,
        verbose=250,
    )

    va_probs = cp.asarray(model.predict_proba(X_va_cp)).astype(cp.float32)
    te_probs = cp.asarray(model.predict_proba(X_te_cp)).astype(cp.float32)
    oof[va_idx] = cp.asnumpy(va_probs)
    oof_fold[va_idx] = fold
    test_pred_sum += cp.asnumpy(te_probs) / N_SPLITS

    fold_score = balanced_accuracy_score(y_cpu[va_idx], np.argmax(oof[va_idx], axis=1))
    best_iter = getattr(model, 'best_iteration', None)
    print(f'fold {fold} balanced accuracy: {fold_score:.8f} | best_iteration={best_iter}')
    fold_rows.append({
        'fold': fold,
        'balanced_accuracy': float(fold_score),
        'best_iteration': int(best_iter) if best_iter is not None else None,
        'n_train': int(X_tr_cp.shape[0]),
        'n_valid': int(X_va_cp.shape[0]),
        'n_features': int(len(features)),
        'n_te_features': int(len(added_te)),
    })

    gain = model.get_booster().get_score(importance_type='gain')
    for i, f in enumerate(features):
        importance_rows.append({'fold': fold, 'feature': f, 'gain': float(gain.get(f'f{i}', 0.0))})

    del model, X_tr, X_va, X_te, X_tr_cp, X_va_cp, X_te_cp, y_tr, y_va, y_tr_cp, y_va_cp
    del sample_weight, valid_weight, va_probs, te_probs
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()

fold_scores = pd.DataFrame(fold_rows)
feature_importance = pd.DataFrame(importance_rows)
cv_score = balanced_accuracy_score(y_cpu, np.argmax(oof, axis=1))
print('\nFold scores:')
display(fold_scores)
print(f'Mean fold balanced accuracy: {fold_scores["balanced_accuracy"].mean():.8f}')
print(f'OOF balanced accuracy: {cv_score:.8f}')


===== Fold 1/5 | seed=142 =====
training shape: (461877, 370) validation shape: (115470, 370) test shape: (247435, 370)
TE features added before top selection: 132
[0]	validation_0-mlogloss:1.08269	validation_0-balanced_error_metric:0.04401
[250]	validation_0-mlogloss:0.13955	validation_0-balanced_error_metric:0.03494
[500]	validation_0-mlogloss:0.09658	validation_0-balanced_error_metric:0.03301
[750]	validation_0-mlogloss:0.09145	validation_0-balanced_error_metric:0.03207
[1000]	validation_0-mlogloss:0.09031	validation_0-balanced_error_metric:0.03172
[1250]	validation_0-mlogloss:0.09002	validation_0-balanced_error_metric:0.03157
[1500]	validation_0-mlogloss:0.09004	validation_0-balanced_error_metric:0.03140
[1750]	validation_0-mlogloss:0.09025	validation_0-balanced_error_metric:0.03137
[1835]	validation_0-mlogloss:0.09034	validation_0-balanced_error_metric:0.03136
fold 1 balanced accuracy: 0.96869380 | best_iteration=1655

===== Fold 2/5 | seed=242 =====
training shape: (461877, 370)

,fold,balanced_accuracy,best_iteration,n_train,n_valid,n_features,n_te_features
0,1,0.968694,1655,461877,115470,370,132
1,2,0.967964,1004,461877,115470,370,132
2,3,0.967654,1221,461878,115469,370,132
3,4,0.966845,890,461878,115469,370,132
4,5,0.967325,1219,461878,115469,370,132


Mean fold balanced accuracy: 0.96769631
OOF balanced accuracy: 0.96769631


## Feature Importance

In [10]:
top_importance = (
    feature_importance.groupby('feature', as_index=False)['gain']
    .mean()
    .sort_values('gain', ascending=False)
    .reset_index(drop=True)
)
display(top_importance.head(50))

,feature,gain
0,TE_art_g_r_half_x_redshift_tenth_QSO,4356.156494
1,TE_art_g_r_half_x_redshift_tenth_STAR,4043.103271
2,orig_art_g_r_half_x_redshift_tenth_prior_QSO,1210.674841
3,redshift_u,1183.946350
4,z_over_redshift,1081.665918
5,TE_art_u_g_half_x_redshift_tenth_QSO,1027.368909
6,TE_art_g_r_half_x_redshift_tenth_GALAXY,1014.754199
7,g_z,972.101550
8,g_i,480.028479
9,orig_g_qbin64_prior_QSO,369.905249


## Save Artifacts

In [11]:
test_preds = test_pred_sum.astype('float32')
oof_preds = oof.astype('float32')

np.save(OOF_PATH, oof_preds)
np.save(PRED_PATH, test_preds)

pred_labels = [INT_TO_CLASS[i] for i in np.argmax(test_preds, axis=1)]
if sample is not None and ID_COL in sample.columns:
    submission = sample.copy()
    submission[TARGET] = pred_labels
else:
    submission = pd.DataFrame({ID_COL: test_ids.to_pandas().to_numpy(), TARGET: pred_labels})
submission.to_csv(SUB_PATH, index=False)

print('Saved exactly three files:')
print(' ', OOF_PATH, oof_preds.shape)
print(' ', PRED_PATH, test_preds.shape)
print(' ', SUB_PATH, submission.shape)
display(submission.head())

Saved exactly three files:
  train_oof/xgb-5_oof.npy (577347, 3)
  test_preds/xgb-5_test_preds.npy (247435, 3)
  xgb-5_submission.csv (247435, 2)


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
